# Day 4: JSON / HTTP API (solution)

Reference solutions. Every cell is runnable; `practice.ipynb` is the blank-page twin.

## `s1_data.py`: Data — navigate nested JSON safely.

In [ ]:
import json
doc = json.loads('{"user":{"name":"Ann","langs":["py","go"],"addr":null}}')

In [ ]:
assert doc["user"]["langs"][-1] == "go"
assert doc["user"]["addr"] is None
assert doc.get("missing", {}).get("x") is None

In [ ]:
name, langs = doc["user"]["name"], doc["user"]["langs"]

In [ ]:
assert (name, len(langs)) == ("Ann", 2)
assert "user.name" in {f"user.{k}": v for k, v in doc["user"].items()}

In [ ]:
assert json.dumps({"b": 1, "a": 2}, sort_keys=True) == '{"a": 2, "b": 1}'

**Then, vary it:**

- VARIATION: write get_path(doc, "user.langs.0") that walks keys/indexes
- VARIATION: dump with indent=2 and separators=(",", ":")

## `s2_flow.py`: Flow — paginated generator, flatten.

In [ ]:
pages = [{"items": [1, 2], "next": 1}, {"items": [3], "next": 2}, {"items": [], "next": None}]
def fetch(i): return pages[i]
def all_items():
    i = 0
    while i is not None:
        p = fetch(i)
        yield from p["items"]
        i = p["next"]

In [ ]:
assert list(all_items()) == [1, 2, 3]

In [ ]:
nested = [[1, 2], [3], []]

In [ ]:
assert [x for sub in nested for x in sub] == [1, 2, 3]

**Then, vary it:**

- VARIATION: stop after the first N items with itertools.islice (lazy!)
- VARIATION: add a max_pages guard to avoid infinite loops

## `s3_functions.py`: Functions — retry decorator, positional-only, kwargs to query.

In [ ]:
import functools
def retry(times=3):
    def deco(fn):
        @functools.wraps(fn)
        def w(*a, **k):
            for attempt in range(times):
                try: return fn(*a, **k)
                except ConnectionError:
                    if attempt == times - 1: raise
        return w
    return deco
calls = []
@retry(3)
def flaky():
    calls.append(1)
    if len(calls) < 3: raise ConnectionError
    return "ok"

In [ ]:
assert flaky() == "ok" and len(calls) == 3

In [ ]:
def build_url(base, /, **params):
    return base + "?" + "&".join(f"{k}={v}" for k, v in params.items())

In [ ]:
assert build_url("http://x", a=1, b=2) == "http://x?a=1&b=2"

**Then, vary it:**

- VARIATION: add exponential backoff via time.sleep(base * 2**attempt) (use tiny base)
- VARIATION: retry only on a tuple of exception types passed in

## `s4_classes.py`: Classes — dependency injection (fake transport), from_json.

In [ ]:
from dataclasses import dataclass
@dataclass
class User:
    id: int
    name: str
    @classmethod
    def from_json(cls, d): return cls(id=d["id"], name=d["name"])
class Client:
    def __init__(self, transport): self._t = transport
    def user(self, id): return User.from_json(self._t(f"/users/{id}"))
fake = lambda path: {"id": int(path.split("/")[-1]), "name": "Ann"}

In [ ]:
assert Client(fake).user(7) == User(7, "Ann")

**Then, vary it:**

- VARIATION: add Client.users(ids) returning a list
- VARIATION: add __repr__ to Client that hides secrets like a token

## `s5_errors.py`: Errors — exception hierarchy, ExceptionGroup.

In [ ]:
class ApiError(Exception):
    def __init__(self, status):
        super().__init__(f"HTTP {status}")
        self.status = status
class NotFound(ApiError): pass
def check(status):
    if status == 404: raise NotFound(status)
    if status >= 400: raise ApiError(status)
    return status

In [ ]:
for s, exc in [(404, NotFound), (500, ApiError)]:
    try: check(s)
    except exc as e: assert e.status == s
assert check(200) == 200

In [ ]:
try:
    raise ExceptionGroup("many", [ValueError("a"), KeyError("b")])
except* ValueError as eg:
    assert len(eg.exceptions) == 1
except* KeyError as eg:
    assert len(eg.exceptions) == 1

**Then, vary it:**

- VARIATION: catch NotFound before ApiError, then swap the order; what changes?
- VARIATION: add .add_note() to an exception and print the traceback

## `s6_stdlib.py`: Stdlib — urllib.parse, HTTPStatus, base64, hashlib, datetime.

In [ ]:
import base64, hashlib
from urllib.parse import urlparse, parse_qs, urlencode
from http import HTTPStatus
from datetime import datetime, timezone
u = urlparse("https://x.com/a/b?q=1&q=2&z=3")

In [ ]:
from urllib.parse import urlparse, parse_qs, urlencode

assert u.netloc == "x.com" and parse_qs(u.query)["q"] == ["1", "2"]

In [ ]:
assert urlencode({"a": 1, "b": "x y"}) == "a=1&b=x+y"
assert HTTPStatus(404).phrase == "Not Found"
assert base64.b64encode(b"hi").decode() == "aGk="
assert hashlib.sha256(b"").hexdigest().startswith("e3b0c442")
assert datetime(2026, 1, 2, tzinfo=timezone.utc).isoformat() == "2026-01-02T00:00:00+00:00"

**Then, vary it:**

- VARIATION: parse an ISO string back with datetime.fromisoformat
- VARIATION: build an HMAC signature with the hmac module

## `s7_concurrency.py`: Concurrency — parallel I/O waits with threads.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
def slow(n):
    time.sleep(0.1)
    return n * 2
t = time.perf_counter()
with ThreadPoolExecutor(5) as ex:
    futs = [ex.submit(slow, i) for i in range(5)]
    got = sorted(f.result() for f in as_completed(futs))

In [ ]:
import time

assert got == [0, 2, 4, 6, 8] and time.perf_counter() - t < 0.4

**Then, vary it:**

- VARIATION: use max_workers=1 and watch the timing assertion fail
- VARIATION: make one task raise; see where the exception surfaces

## `s8_quality.py`: Quality — generics, Literal, Callable, match on types.

In [ ]:
from typing import Callable, Literal, TypeVar
T = TypeVar("T")
def first(xs: list[T], default: T) -> T: return xs[0] if xs else default

In [ ]:
assert first([], 0) == 0 and first(["a"], "z") == "a"

In [ ]:
Method = Literal["GET", "POST"]
def req(m: Method, h: Callable[[str], str]) -> str: return h(m)

In [ ]:
assert req("GET", str.lower) == "get"

In [ ]:
def parse(v: int | str | None) -> int:
    match v:
        case None: return 0
        case int(): return v
        case str(): return int(v)

In [ ]:
assert (parse(None), parse(3), parse("4")) == (0, 3, 4)

**Then, vary it:**

- VARIATION: rewrite `first` with PEP 695 syntax: def first[T](...)
- VARIATION: if mypy is installed, run it and break a type on purpose

## `kata.py`: KATA — real HTTP round-trip against a local server.

In [ ]:
import json, threading, urllib.request
from http.server import BaseHTTPRequestHandler, HTTPServer
class H(BaseHTTPRequestHandler):
    def do_GET(self):
        body = json.dumps({"path": self.path}).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(body)
    def log_message(self, *a): pass
srv = HTTPServer(("127.0.0.1", 0), H)
threading.Thread(target=srv.serve_forever, daemon=True).start()
def get_json(path):
    with urllib.request.urlopen(f"http://127.0.0.1:{srv.server_port}{path}", timeout=2) as r:
        return json.load(r)

In [ ]:
assert get_json("/users/1") == {"path": "/users/1"}

In [ ]:
srv.shutdown()

**Then, vary it:**

- KATA: make the server return 404 for /missing and map it to NotFound (see s5)